### Доступ к НКРЯ через API

API (application programming interface) &mdash; это общее название для способов общения между программами. Часто этим словом обозначают *web API* &mdash; то есть API для веб-сервера.

Для обращения к API используют так называемые HTTP-запросы &mdash; структурированные сообщения. Они используют разные методы в зависимости от желаемого действия. Чаще всего это методы `GET` для получения данных с сервера и `POST` для отправки данных на сервер.

Для работы с HTTP-запросами в Python наиболее удобное средство &mdash; сторонняя библиотека `requests`. Её можно установить через командную строку: `python -m pip install requests`, `py -m pip install requests` или просто `pip install requests`. В Google Colab она установлена по умолчанию.

Документация по API НКРЯ расположена по двум основным адресам:

https://ruscorpora.ru/api/v1/docs

https://ruscorpora.github.io/public-api/index.html

Для обращения к НКРЯ запросы мы будем посылать на адреса, начинающиеся на https://ruscorpora.ru/api/v1. Чтобы передать запрос GET, вызовем функцию `get`. Она принимает на вход адрес и ряд дополнительных аргументов, из которых нас будут интересовать `params` и `headers`.

В аргумент `params` мы должны будем передать словарь, ключи которого &mdash; названия параметров, требуемые API, а значения &mdash; строки с нужными данными. API НКРЯ принимает данные в формате JSON, поэтому нам потребуется также библиотека `json`.

В аргумент `headers` передадим также словарь, в котором должен быть ключ `"Authorization"` и значение `"Bearer YOUR_API_KEY`, где на месте `YOUR_API_KEY` должен стоять ваш ключ доступа в API НКРЯ.

Этот ключ &mdash; секретная информация, поэтому нужно держать его в секрете. Проще всего завести отдельный файл (например, *key.py*), в котором завести ключ в переменную и импортировать его оттуда. Этот файл нельзя загружать в какие-либо репозитории или отправлять другим лицам.

In [1]:
import requests
import json

In [ ]:
api_key = "YOUR_API_KEY"

Я храню свой ключ на гугл диске:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/key.py .

In [ ]:
from key import key as api_key

Напишем запрос для получения данных об основном подкорпусе НКРЯ. Этот запрос отправляется на адрес https://ruscorpora.ru/api/v1/stats. Данные пришли в виде специального объекта, информацию из которого мы можем извлечь через метод `.json()`, который вернёт словарь или список словарей, пригодный для записи в JSON.

In [12]:
url = "https://ruscorpora.ru/api/v1/stats"

params = {
    "corpus": json.dumps({
        "type": "MAIN"
    }, ensure_ascii=False)
}
headers = {
    "Authorization": f"Bearer {api_key}"
}
response = requests.get(url, params=params, headers=headers)
data = response.json()

print(json.dumps(data, indent=2))

{
  "corpus": {
    "type": "MAIN",
    "lang": null
  },
  "vals": {
    "textCount": 133554,
    "sentenceCount": 31626104,
    "wordUsageCount": 389471513
  },
  "subStats": [
    {
      "disamb": [
        "MANUALLY_DISAMB"
      ],
      "vals": {
        "textCount": 2164,
        "sentenceCount": 514169,
        "wordUsageCount": 5988177
      }
    }
  ]
}


Поисковые запросы для лексико-грамматического поиска отправляются на адрес https://ruscorpora.ru/api/v1/lex-gramm/concordance. Построим запрос: найдём слово *кот* на расстоянии от -1 до 1 от глагола. 

https://ruscorpora.github.io/public-api/about-api/search/lex-gramm.html

In [ ]:
query = {
    "corpus": {
        "type": "MAIN"
    },
    "subcorpus": {  # подкорпус - эту часть можно опустить, тогда поиск будет по всему корпусу
        "sectionValues": [
            {
                "conditionValues": [
                    {
                        "fieldName": "pri_cat:нехудожественная",
                        "check": {
                            "v": True
                        }
                    },
                    {
                        "fieldName": "sphere:нехудожественная",
                        "text": {
                            "v": "публицистика"
                        }
                    }
                ]
            }
        ]
    },
    "lexGramm": {
        "sectionValues": [
            {
                "subsectionValues": [
                    {
                        "conditionValues": [
                            {
                                "fieldName": "lex",
                                "text": {
                                    "v": "кот"
                                }
                            }
                        ]
                        },
                    {
                        "conditionValues": [
                            {
                                "fieldName": "gramm",  # название поля - смотрим в структуре запроса (ниже)
                                "text": {
                                    "v": "V"  # значение поля - совпадает с веб-версией
                                }
                            },
                            {
                                "fieldName": "dist",
                                "intRange": {
                                    "begin": -1,
                                    "end": 1
                                }
                            }
                        ]
                    }
                ]
            }
        ]
    },
    "params": {
        "pageParams": {
            "page": 0  # сюда передаём номер страницы
        }
    }
}

params = {
    "query": json.dumps(query, ensure_ascii=False),
    "corpus": json.dumps({"type": "MAIN"})
}

In [54]:
url = "https://ruscorpora.ru/api/v1/lex-gramm/concordance"

response = requests.get(url, params=params, headers=headers)
data = response.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "corpusStats": {
    "textCount": "133554",
    "sentenceCount": "31626104",
    "wordUsageCount": "389471513"
  },
  "subcorpStats": {
    "textCount": "98350",
    "wordUsageCount": "140263052"
  },
  "queryStats": {
    "textCount": "490",
    "wordUsageCount": "1265"
  },
  "groups": [
    {
      "docs": [
        {
          "info": {
            "title": "Алексей Володин. Чечевичная похлебка. «Мумий Тролль» выступит в Лужниках // «Известия», 2002.08.29",
            "source": {
              "docId": "standard/post1950/izvestia/23132.xml"
            },
            "homonymyStatus": "manual",
            "allExamplesNum": 1,
            "docExplainInfo": {
              "items": [
                {
                  "parsingFields": [
                    {
                      "name": "author",
                      "value": [
                        {
                          "valString": {
                            "v": "Алексей Володин"
                          }
   

В вернувшемся объекте сложная структура. Познакомимся с ней поближе.

In [20]:
print("\n".join(sorted(data.keys())))

corpusStats
groups
pagination
queryStats
sorting
subcorpStats


`data["groups"]` &mdash; список из &laquo;групп&raquo; &mdash; результатов поиска. Каждая группа &mdash; словарь. У неё один ключ &mdash; `"docs"`.

In [52]:
data["groups"][0]["docs"][0]["info"]

{'title': 'Наши дети: Подростки (2004)',
 'source': {'docId': 'standard/post1950/forum/103.xml'},
 'homonymyStatus': 'manual',
 'allExamplesNum': 1,
 'docExplainInfo': {'items': [{'parsingFields': [{'name': 'author',
      'value': [{'valString': {'v': 'коллективный'}}]},
     {'name': 'header',
      'value': [{'valString': {'v': 'Наши дети: Подростки'}}]},
     {'name': 'created', 'value': [{'valString': {'v': '2004'}}]},
     {'name': 'sphere',
      'value': [{'valString': {'v': 'электронная коммуникация'}}]},
     {'name': 'topic', 'value': [{'valString': {'v': 'частная жизнь'}}]},
     {'name': 'type', 'value': [{'valString': {'v': 'форум'}}]},
     {'name': 'style', 'value': [{'valString': {'v': 'нейтральный'}}]},
     {'name': 'sentences', 'value': [{'valString': {'v': '3471'}}]},
     {'name': 'words', 'value': [{'valString': {'v': '37531'}}]},
     {'name': 'audience_age', 'value': [{'valString': {'v': 'н-возраст'}}]},
     {'name': 'audience_level', 'value': [{'valString': {

In [23]:
print("\n".join(sorted(data["groups"][0]["docs"][0].keys())))

info
snippetGroups


In [55]:
data["groups"][0]["docs"][0]["info"]

{'title': 'Алексей Володин. Чечевичная похлебка. «Мумий Тролль» выступит в Лужниках // «Известия», 2002.08.29',
 'source': {'docId': 'standard/post1950/izvestia/23132.xml'},
 'homonymyStatus': 'manual',
 'allExamplesNum': 1,
 'docExplainInfo': {'items': [{'parsingFields': [{'name': 'author',
      'value': [{'valString': {'v': 'Алексей Володин'}}]},
     {'name': 'sex', 'value': [{'valString': {'v': 'муж'}}]},
     {'name': 'header',
      'value': [{'valString': {'v': 'Чечевичная похлебка. «Мумий Тролль» выступит в Лужниках'}}]},
     {'name': 'created', 'value': [{'valString': {'v': '2002'}}]},
     {'name': 'sphere', 'value': [{'valString': {'v': 'публицистика'}}]},
     {'name': 'topic',
      'value': [{'valString': {'v': 'искусство и культура'}}]},
     {'name': 'type', 'value': [{'valString': {'v': 'статья'}}]},
     {'name': 'style', 'value': [{'valString': {'v': 'нейтральный'}}]},
     {'name': 'sentences', 'value': [{'valString': {'v': '22'}}]},
     {'name': 'words', 'value'

In [59]:
data["groups"][0]["docs"][0]["snippetGroups"]

[{'snippets': [{'sequences': [{'words': [{'type': 'WORD',
        'text': 'Через',
        'source': {'sentId': 14},
        'displayParams': {}},
       {'type': 'PLAIN', 'text': ' ', 'source': {}, 'displayParams': {}},
       {'type': 'WORD',
        'text': 'две',
        'source': {'sentId': 14, 'wordId': 1},
        'displayParams': {}},
       {'type': 'PLAIN', 'text': ' ', 'source': {}, 'displayParams': {}},
       {'type': 'WORD',
        'text': 'недели',
        'source': {'sentId': 14, 'wordId': 2},
        'displayParams': {}},
       {'type': 'PLAIN', 'text': ' ', 'source': {}, 'displayParams': {}},
       {'type': 'WORD',
        'text': 'песня',
        'source': {'sentId': 14, 'wordId': 3},
        'displayParams': {}},
       {'type': 'PLAIN', 'text': ' «', 'source': {}, 'displayParams': {}},
       {'type': 'WORD',
        'text': 'Кот',
        'source': {'sentId': 14, 'wordId': 4},
        'displayParams': {}},
       {'type': 'PLAIN', 'text': ' ', 'source': {}, 'di

In [57]:
"".join(w["text"] for w in data["groups"][0]["docs"][0]["snippetGroups"][0]["snippets"][0]["sequences"][0]["words"])

'Через две недели песня «Кот кота» звучала отовсюду. '

Чтобы получить структуру поисковой формы, мы можем отправить запрос на адрес https://ruscorpora.ru/api/v1/lex-gramm/search-form.

In [42]:
url = "https://ruscorpora.ru/api/v1/lex-gramm/search-form"

params = {
    "corpus": json.dumps({
        "type": "MAIN"
    }, ensure_ascii=False)
}

response = requests.get(url, params=params, headers=headers)
data = response.json()

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "lexGramm": [
    {
      "conditions": [
        {
          "fieldName": "disambmod",
          "showByDefault": true,
          "radioGroup": {
            "variants": [
              {
                "variant": {
                  "title": "all_parsings",
                  "value": "all"
                }
              },
              {
                "variant": {
                  "title": "main_parsing",
                  "value": "main"
                },
                "default": true
              }
            ]
          }
        },
        {
          "fieldName": "distmod",
          "showByDefault": true,
          "radioGroup": {
            "variants": [
              {
                "variant": {
                  "title": "no_zeros",
                  "value": "no_zeros"
                }
              },
              {
                "variant": {
                  "title": "with_zeros",
                  "value": "with_zeros"
                },
          

Если вы хотите сделать сразу много запросов к API, может быть полезно сделать перерывы между ними с помошью функции `time.sleep()`, которая принимает один аргумент &mdash; время паузы в секундах.

In [60]:
import time

print("1")
time.sleep(1)
print("2")

1
2


### Задания для самостоятельного выполнения

#### Задание 1

Запишите структуру лексико-грамматического поиска в файл JSON и изучите поля `conditionValues`.

#### Задание 2

Измените запрос так, чтобы находились сочетания "глагол в форме первого лица единственного числа настоящего или будущего времени" + словоформа "котом". Запишите первые две страницы результатов в текстовый файл, отсортировав по найденному глаголу.

#### Задание 3

Найдите все примеры сочетаний "ждать" + "сущ. в род. п." и "ждать" + "сущ. в вин. п." в каком-нибудь подкорпусе (задайте его сами). Постройте график зависимости того и другого от года. 

#### Задание 4

Постройте частотный словарь существительных в винительном падеже, встречающихся после слова "видеть" в каком-нибудь подкорпусе (задайте его сами). Используйте `pymorphy` для получение нормальной формы.


### Домашнее задание

Нет :) Можно доделать задания с пары (по желанию).